# Business Entity Resolution Pipeline — Kaggle Runner (Memory & Parallelism Optimized)

This notebook runs the complete entity resolution pipeline in Kaggle with safe CPU parallelism and strict memory hygiene.

### Recommended Kaggle Settings:
- **Accelerator:** GPU P100 or GPU T4 x2 (for multilingual sentence embedding acceleration)
- **Internet:** ON (for cloning repo, downloading dependencies, and Hugging Face weights)
- **Persistence:** Files only (optional)

## 1. Environment, Hardware & Parallelism Configuration

In [ ]:
import os
import gc
import sys
import time
import torch

# Safe CPU parallelism: Bounded to avoid CPU oversubscription and memory thrashing
NUM_WORKERS = max(1, min(4, os.cpu_count() or 2))
print(f"Safe CPU workers configured: {NUM_WORKERS} (system CPUs: {os.cpu_count()})")

# Check CUDA hardware
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
    print(f"Initial VRAM Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
else:
    print("CUDA not available; falling back to CPU.")

# Memory hygiene helper: clears unreferenced objects and flushes unused GPU cache
def flush_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

## 2. Clone Repository & Install Dependencies

In [ ]:
# Work inside /kaggle/working
%cd /kaggle/working

if not os.path.exists("amazon-ml"):
    !git clone https://github.com/tamcee/amazon-ml.git

repo_path = "/kaggle/working/amazon-ml/student_resource/code/business_entity_resolution"
%cd {repo_path}
!git pull origin main

# Add repository directory to sys.path so modules can be imported directly in notebook
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# Install pinned dependencies
!pip install -q -r requirements.txt

## 3. Dataset Configuration & Output Setup

In [ ]:
# Auto-detect dataset directory under /kaggle/input
input_root = "/kaggle/input"
train_dir = None
test_dir = None

for root, dirs, files in os.walk(input_root):
    if "train_source1.tsv" in files and train_dir is None:
        train_dir = root
    if "test_source1.tsv" in files and test_dir is None:
        test_dir = root

# Fallback check for local repository dataset directory
if not train_dir and os.path.exists("../../dataset/train"):
    train_dir = os.path.abspath("../../dataset/train")
if not test_dir and os.path.exists("../../dataset/test"):
    test_dir = os.path.abspath("../../dataset/test")

output_dir = "/kaggle/working/output"
os.makedirs(output_dir, exist_ok=True)

if train_dir:
    os.environ['TRAIN_DIR'] = train_dir
    print(f"Detected TRAIN_DIR: {train_dir}")
else:
    print("WARNING: train_source1.tsv not found in /kaggle/input. Please set os.environ['TRAIN_DIR'] manually.")

if test_dir:
    os.environ['TEST_DIR'] = test_dir
    print(f"Detected TEST_DIR:  {test_dir}")
else:
    print("WARNING: test_source1.tsv not found in /kaggle/input. Please set os.environ['TEST_DIR'] manually.")

os.environ['OUTPUT_DIR'] = output_dir
print(f"Set OUTPUT_DIR:    {output_dir}")

## 4. Optional: Quick Smoke Test with Dev Cohort

In [ ]:
# Set DEV_TEST = True to run a quick 5,000-entity verification before full training
DEV_TEST = False

if DEV_TEST:
    from src.create_dev_cohort import create_dev_cohort
    from run_pipeline import stage_train
    import argparse
    
    print("Running smoke test on 5,000-entity dev cohort...")
    create_dev_cohort(n_entities=5000, output_dir="artifacts/dev_cohort")
    os.environ['TRAIN_DIR'] = "artifacts/dev_cohort"
    stage_train(argparse.Namespace())
    
    # Restore full dataset path after dev test
    os.environ['TRAIN_DIR'] = train_dir
    flush_memory()

## 5. Stage 1: Full-Scale Training Pipeline

Executes full training with step-by-step memory management, freeing intermediate DataFrames, clearing GPU VRAM, and training LightGBM with bounded thread concurrency.

In [ ]:
import pandas as pd
import numpy as np

from src.config import cfg, TRAIN_DIR, ARTIFACTS_DIR, DEVICE
from src.normalize import normalize_all_sources
from src.blocking import run_blocking
from src.embeddings import EmbeddingManager
from src.features import FeatureExtractor
from src.train import train_model, _grouped_split, FEATURE_COLS
from src.threshold import sweep_threshold
from run_pipeline import load_ground_truth

t_train_start = time.time()
active_train_dir = os.environ.get('TRAIN_DIR', TRAIN_DIR)
print(f"Training on data from: {active_train_dir}")

# 1. Load training records & ground truth
print("\n--- 1. Loading Training Data ---")
s1 = pd.read_csv(os.path.join(active_train_dir, 'train_source1.tsv'), sep='\t', dtype=str, keep_default_na=False)
s2 = pd.read_csv(os.path.join(active_train_dir, 'train_source2.tsv'), sep='\t', dtype=str, keep_default_na=False)
s3 = pd.read_csv(os.path.join(active_train_dir, 'train_source3.tsv'), sep='\t', dtype=str, keep_default_na=False)
gt = load_ground_truth(os.path.join(active_train_dir, 'train_ground_truth.tsv'))
print(f"Loaded: S1={len(s1)}, S2={len(s2)}, S3={len(s3)}, GT={len(gt)}")

# 2. Text Normalization (cached to parquet if previously computed)
print("\n--- 2. Normalizing Text Fields ---")
norm_cache = os.path.join(ARTIFACTS_DIR, 'train_normalized')
s1, s2, s3 = normalize_all_sources(s1, s2, s3, cache_dir=norm_cache)
flush_memory()

# 3. Candidate Generation (Blocking with incremental pruning)
print("\n--- 3. Running Multi-Strategy Blocking ---")
candidates = run_blocking(
    s1, s2, s3,
    max_block_size=cfg['blocking']['max_block_size'],
    top_k=cfg['blocking']['top_k_per_entity'],
    ground_truth=gt
)
flush_memory()

# 4. Multilingual Embeddings Generation
print("\n--- 4. Computing Multilingual Embeddings ---")
emb_mgr = EmbeddingManager(
    cache_dir=os.path.join(ARTIFACTS_DIR, 'train_embeddings'),
    batch_size=cfg['features']['embedding_batch_size'],
    device=DEVICE
)
# Concatenate S2 and S3 for embedding lookup and free individual dataframes
s23 = pd.concat([s2, s3], ignore_index=True)
del s2, s3
flush_memory()

name_embs, addr_embs = emb_mgr.embed_candidate_pairs(s1, s23, candidates)
# Release PyTorch GPU memory after encoding
flush_memory()

# 5. Sparse TF-IDF & Pairwise Feature Extraction
print("\n--- 5. Extracting Pairwise Features ---")
feat_ext = FeatureExtractor(max_features=cfg['features']['tfidf_max_features'])

# Fit TF-IDF on name series without creating redundant full DataFrame copies
name_col = 'norm_business_name' if 'norm_business_name' in s1.columns else 'business_name'
all_names = pd.concat([s1[name_col], s23[name_col]], ignore_index=True)
all_ids = pd.concat([s1['entity_id'], s23['entity_id']], ignore_index=True)
feat_ext.fit_tfidf(all_names, all_ids)
del all_names, all_ids
flush_memory()

features_df = feat_ext.extract_all_features(s1, s23, candidates, name_embs, addr_embs)

# Free candidate embeddings and s23 as features are now fully extracted
del name_embs, addr_embs, s23, s1, candidates
flush_memory()

# Add binary match labels via fast itertuples
labels = []
for row in features_df[['s1_id', 's23_id']].itertuples(index=False):
    labels.append(1 if row.s23_id in gt.get(row.s1_id, set()) else 0)
features_df['label'] = labels
print(f"Features ready: {len(features_df)} pairs ({sum(labels)} positive, {len(labels) - sum(labels)} negative)")

# 6. Model Training & Monotonic Threshold Sweep
print("\n--- 6. Training LightGBM Model ---")
# Configure parallel jobs for LightGBM bounded by NUM_WORKERS
cfg['model']['n_jobs'] = NUM_WORKERS

model, best_tau, val_f05, importance = train_model(features_df, gt)

# Perform threshold optimization sweep on validation candidates
_, val_df = _grouped_split(features_df)
sweep_results = sweep_threshold(val_df, gt, model=model)
optimal_tau = sweep_results['best']['tau']

# Free training features and ground truth mappings
del features_df, val_df, gt
flush_memory()

print(f"\nTraining completed in {time.time() - t_train_start:.1f}s")
print(f"Optimal Threshold: {optimal_tau:.2f} | Best Val F0.5: {sweep_results['best']['f05']:.4f}")

## 6. Stage 2: Full-Scale Test Inference Pipeline

Runs end-to-end inference on the test set: normalization, candidate blocking, embedding generation, feature extraction, scoring, thresholding, and format-compliant file generation.

In [ ]:
from src.config import cfg, TEST_DIR, OUTPUT_DIR, ARTIFACTS_DIR, DEVICE
from src.normalize import normalize_all_sources
from src.blocking import run_blocking
from src.embeddings import EmbeddingManager
from src.features import FeatureExtractor
from src.train import load_model, FEATURE_COLS
from src.infer import _write_matching_results, _write_candidate_pairs

t_infer_start = time.time()
active_test_dir = os.environ.get('TEST_DIR', TEST_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Running test inference on data from: {active_test_dir}")

# Ensure model and threshold are loaded
if 'model' not in locals() or model is None:
    model, meta = load_model()
    optimal_tau = meta.get('best_threshold', 0.5)

# 1. Load test records
print("\n--- 1. Loading Test Records ---")
s1_test = pd.read_csv(os.path.join(active_test_dir, 'test_source1.tsv'), sep='\t', dtype=str, keep_default_na=False)
s2_test = pd.read_csv(os.path.join(active_test_dir, 'test_source2.tsv'), sep='\t', dtype=str, keep_default_na=False)
s3_test = pd.read_csv(os.path.join(active_test_dir, 'test_source3.tsv'), sep='\t', dtype=str, keep_default_na=False)
print(f"Test Records: S1={len(s1_test)}, S2={len(s2_test)}, S3={len(s3_test)}")

# 2. Normalize text fields
print("\n--- 2. Normalizing Test Data ---")
test_norm_cache = os.path.join(ARTIFACTS_DIR, 'test_normalized')
s1_test, s2_test, s3_test = normalize_all_sources(s1_test, s2_test, s3_test, cache_dir=test_norm_cache)
flush_memory()

# 3. Candidate Generation (Blocking)
print("\n--- 3. Blocking Test Records ---")
test_candidates = run_blocking(
    s1_test, s2_test, s3_test,
    max_block_size=cfg['blocking']['max_block_size'],
    top_k=cfg['blocking']['top_k_per_entity']
)

# Immediately persist candidate_pairs.tsv
_write_candidate_pairs(test_candidates, OUTPUT_DIR)
flush_memory()

# 4. Multilingual Embeddings Generation
print("\n--- 4. Computing Test Embeddings ---")
emb_mgr_test = EmbeddingManager(
    cache_dir=os.path.join(ARTIFACTS_DIR, 'test_embeddings'),
    batch_size=cfg['features']['embedding_batch_size'],
    device=DEVICE
)
s23_test = pd.concat([s2_test, s3_test], ignore_index=True)
del s2_test, s3_test
flush_memory()

name_embs_test, addr_embs_test = emb_mgr_test.embed_candidate_pairs(s1_test, s23_test, test_candidates)
# Free PyTorch GPU cache after encoding
flush_memory()

# 5. Feature Extraction
print("\n--- 5. Extracting Test Features ---")
feat_ext_test = FeatureExtractor(max_features=cfg['features']['tfidf_max_features'])

name_col_test = 'norm_business_name' if 'norm_business_name' in s1_test.columns else 'business_name'
all_test_names = pd.concat([s1_test[name_col_test], s23_test[name_col_test]], ignore_index=True)
all_test_ids = pd.concat([s1_test['entity_id'], s23_test['entity_id']], ignore_index=True)
feat_ext_test.fit_tfidf(all_test_names, all_test_ids)
del all_test_names, all_test_ids
flush_memory()

test_features_df = feat_ext_test.extract_all_features(
    s1_test, s23_test, test_candidates, name_embs_test, addr_embs_test
)
# Free raw embeddings, s23_test, and candidate mappings
del name_embs_test, addr_embs_test, s23_test, test_candidates
flush_memory()

# 6. Model Scoring & Vectorized Prediction Formatting
print("\n--- 6. Scoring Test Candidates ---")
X_test = test_features_df[FEATURE_COLS].values
test_features_df['proba'] = model.predict(X_test)
del X_test
flush_memory()

print(f"Applying optimal threshold tau={optimal_tau:.2f}...")
predictions = {s1_id: set() for s1_id in s1_test['entity_id'].unique()}

# Filter candidates passing threshold and populate predictions via itertuples
above = test_features_df[test_features_df['proba'] >= optimal_tau]
for row in above[['s1_id', 's23_id']].itertuples(index=False):
    predictions[row.s1_id].add(row.s23_id)

del test_features_df, above, s1_test
flush_memory()

# 7. Write Final matching_results.tsv
_write_matching_results(predictions, OUTPUT_DIR)
del predictions
flush_memory()

print(f"\nInference completed in {time.time() - t_infer_start:.1f}s")

## 7. Official Submission Format Validation

In [ ]:
validator_path = "../../utils/validate_submission.py"
matching_file = os.path.join(os.environ['OUTPUT_DIR'], "matching_results.tsv")
candidate_file = os.path.join(os.environ['OUTPUT_DIR'], "candidate_pairs.tsv")
test_dir_path = os.environ['TEST_DIR']

!python {validator_path} \
    --matching {matching_file} \
    --candidate {candidate_file} \
    --test-dir {test_dir_path}

## 8. Package Final Submission Zip for Download

In [ ]:
output_zip = "/kaggle/working/submission_results.zip"
matching_path = os.path.join(os.environ['OUTPUT_DIR'], "matching_results.tsv")
candidate_path = os.path.join(os.environ['OUTPUT_DIR'], "candidate_pairs.tsv")

!ls -lh {os.environ['OUTPUT_DIR']}
!zip -j {output_zip} {matching_path} {candidate_path}
print(f"\nFinal submission package ready at: {output_zip}")